## Тестовое задание Avito

In [57]:
# импорт библиотек и считывание данных + 5 строчек из train
import pandas as pd
import numpy as np

train = pd.read_parquet("dataset/train.parquet")
queries = pd.read_parquet("dataset/benchmark_queries.parquet")
items = pd.read_parquet("dataset/benchmark_items.parquet")
train.head()

,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,скупка телевизоров,652430,0,,114,Скупка б/у техники,91.0,4.989011,1.000000000000000,2303374,39.712818145751953,652430,54.629230499267578,True,False,Вид услуги Место оказания услуг Первомайский п...,e8b685dffe1a408e,Скупаю практически любую современную новую и б...,114
1,автоподбор,640860,0,Рейтинг пользователя 4 звезды и выше,114,Автоподбор Разовый осмотр автомобиля,462.0,4.982684,3500.000000000000000,2303374,44.051009999999998,640860,56.273389999999999,False,False,Вид услуги Место оказания услуг Нижний Новгоро...,92e1b0446f827b59,🚗 Автоподбор и выездная диагностика автомобиля...,114
2,баня на дровах,653240,0,"Онлайн-запись Тип услуги СПА-услуги, массаж Ви...",114,"Баня на дровах ""Прованс"" на Цветочной",NaN,NaN,1600.000000000000000,86469,30.128784180000000,653240,59.783687590000000,False,False,"Вид услуги Красота, здоровье Место оказания ус...",624846856ce81d69,В ритме современной жизни так сложно найти мо...,114
3,изготовление госномера на авто,634670,0,"Вид услуги Оборудование, производство",114,"Изготовление дубликатов авто номеров, гос номеров",14.0,4.714286,1700.000000000000000,2303428,40.537841000000000,633570,45.424875000000000,False,False,"Вид услуги Оборудование, производство Тип услу...",45b8628b9c6e7b85,Изготовим дубликат номера по утере или износу ...,114
4,укладка плитки,658430,0,Тип услуги Ремонт квартир и домов под ключ Вид...,114,Ремонт и отделка квартир под ключ,1.0,5.000000,1000.000000000000000,44725,69.496444699999998,658430,56.105983729999998,False,False,Вид услуги Ремонт и отделка Тип услуги Ремонт ...,c95a4a7daf2a967f,отделочные работы любой сложности.,114


In [58]:
queries.head() # 5 строк из benchmark_queries

,query_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category
0,70DfDUpwjxB4lzFd,перевозки владикавказ тбилиси,649820,0,,114
1,JTrdTaZJvSiLPkXj,обзвон по базе,107620,0,Вид услуги Деловые услуги,114
2,LZCZNoVG4AFUkVRJ,липоредукция подбородка,637640,0,"Вид услуги Красота, здоровье",114
3,660ac9QVtXkRxZC3,подъемник 4 х стоечный,662810,0,,114
4,YgHcM9MVbxKnxD1e,монтаж видеодомофонов,642790,0,,114


In [59]:
# проверяем датасеты на наличие пропусков и дупликатов
print("Пропуски: ")
print(train.columns.isna(),train.columns.isna().sum()) 
print(queries.columns.isna(),queries.columns.isna().sum())
print("\n")
print("Дупликаты:")
print(train.columns.duplicated())
print(queries.columns.duplicated())

Пропуски: 
[False False False False False False False False False False False False
 False False False False False False False] 0
[False False False False False False] 0


Дупликаты:
[False False False False False False False False False False False False
 False False False False False False False]
[False False False False False False]


#### **Сделаем преобразование данных перед построением кандидатогенератора** 

In [60]:
import re

# 1. Функция для очистки и объединения текста
def clean_text_basic(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-zа-я0-9\s]', '', text)
    return text

#2. Функция объединения текста объявления
def prepare_item_text_basic(row):
    parts = [
        clean_text_basic(row['item_title_raw']),
        clean_text_basic(row['item_description_raw']),
        clean_text_basic(row['item_infm_params_text'])
    ]
    return " ".join(parts)

#3. Функция объединения текста запроса
def prepare_query_text_basic(row):
    parts = [
        clean_text_basic(row['search_query']),
        clean_text_basic(row['search_infm_params_text'])
    ]
    return " ".join(parts)
items['text_word'] = items.apply(prepare_item_text_basic, axis=1)

# Создаем уникальный ключ запроса: сам текст + фильтры + категория + локация

train['query_key'] = (
    train['search_query'].fillna('') + '|' +
    train['search_infm_params_text'].fillna('') + '|' +
    train['search_category'].astype(str) + '|' +
    train['search_location_id'].astype(str)
)

# для каждого уникального запроса собираем список релевантных item_id
train_grouped = train.groupby('query_key').agg({
    'search_query': 'first',
    'search_infm_params_text': 'first',
    'search_category': 'first',
    'search_location_id': 'first',
    'item_id': lambda x: list(set(x))  # уникальные item_id
}).reset_index()


EVAL_SAMPLE_SIZE = 5000
eval_sample = train_grouped.sample(n=EVAL_SAMPLE_SIZE, random_state=42).reset_index(drop=True)
eval_sample['text_word'] = eval_sample.apply(prepare_query_text_basic, axis=1)


In [61]:
# Группировка items по категориям
items_by_cat = {
    int(cat): grp.reset_index(drop=True)
    for cat, grp in items.groupby('item_category_id')
}
print(f"Категорий: {len(items_by_cat)}, медиана размера: {np.median([len(g) for g in items_by_cat.values()]):.0f}")

Категорий: 47, медиана размера: 18


#### **Первый вариант классический - обучить TF-IDF на косинусном сходстве**

In [64]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import tqdm

word_vectorizers = {}
word_matrices = {}

for cat, grp in tqdm.tqdm(items_by_cat.items(), desc="TF-IDF по категориям"):
    if len(grp) < 50:
        continue
    max_feat = min(30000, max(1000, len(grp) // 2))
    vec = TfidfVectorizer(
        max_features=max_feat,
        ngram_range=(1, 1),  # пока только униграммы
        min_df=2,
        sublinear_tf=True
    )
    mat = vec.fit_transform(grp['text_word'])
    word_vectorizers[cat] = vec
    word_matrices[cat] = normalize(mat, axis=1)

# ---------- Функция оценки ----------
def evaluate_retrieval(eval_df, search_fn, k=50, desc="Eval"):
    """
    search_fn(query_row) -> list of top-k item_ids
    """
    recalls = []
    for _, row in tqdm.tqdm(eval_df.iterrows(), total=len(eval_df), desc=desc):
        pred_ids = set(search_fn(row)[:k])
        relevant = set(row['item_id'])
        if not relevant:
            recalls.append(0.0)
        else:
            recalls.append(len(pred_ids & relevant) / len(relevant))
    return np.mean(recalls), recalls

def search_word_tfidf(row, k=50):
    cat = int(row['search_category'])
    if cat not in word_matrices:
        return []
    vec = word_vectorizers[cat]
    mat = word_matrices[cat]
    grp = items_by_cat[cat]
    
    q_vec = normalize(vec.transform([row['text_word']]), axis=1)
    sims = (q_vec @ mat.T).toarray().flatten()
    
    n = len(sims)
    if n == 0:
        return []
    k_eff = min(k, n)
    top_idx = np.argpartition(sims, -k_eff)[-k_eff:]
    top_idx = top_idx[np.argsort(sims[top_idx])[::-1]]
    return grp.iloc[top_idx]['item_id'].tolist()

# ---------- Прогон ----------
recall_word, _ = evaluate_retrieval(eval_sample, search_word_tfidf, desc="Word TF-IDF + cat")
print(f"\n>>> Recall@50 (word TF-IDF + category): {recall_word:.4f}")

Word TF-IDF + cat: 100%|██████████| 5000/5000 [41:14<00:00,  2.02it/s]


>>> Recall@50 (word TF-IDF + category): 0.0099


#### **Поиск топ-50 и подсчет Recall@50**

In [66]:
# Строим символьный TF-IDF для каждой категории
char_vectorizers = {}
char_matrices = {}

for cat, grp in tqdm.tqdm(items_by_cat.items(), desc="Char TF-IDF по категориям"):
    if len(grp) < 50:
        continue
    vec = TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3, 5),
        max_features=50000,
        min_df=3,
        sublinear_tf=True
    )
    mat = vec.fit_transform(grp['text_word'])
    char_vectorizers[cat] = vec
    char_matrices[cat] = normalize(mat, axis=1)

print(f"Готово символьных векторизаторов: {len(char_vectorizers)}")


def search_char_tfidf(row, k=200):
    """Поиск по символьному TF-IDF в пределах категории запроса."""
    cat = int(row['search_category'])
    if cat not in char_matrices:
        return []
    vec = char_vectorizers[cat]
    mat = char_matrices[cat]
    grp = items_by_cat[cat]
    
    q_vec = normalize(vec.transform([row['text_word']]), axis=1)
    sims = (q_vec @ mat.T).toarray().flatten()
    
    n = len(sims)
    if n == 0:
        return []
    k_eff = min(k, n)
    top_idx = np.argpartition(sims, -k_eff)[-k_eff:]
    top_idx = top_idx[np.argsort(sims[top_idx])[::-1]]
    return grp.iloc[top_idx]['item_id'].tolist()


def rrf_fuse(list_of_rankings, k=60):
    """
    Reciprocal Rank Fusion.
    list_of_rankings: список упорядоченных списков item_id.
    k: параметр сглаживания (обычно 60).
    """
    scores = {}
    for ranking in list_of_rankings:
        for rank, item_id in enumerate(ranking):
            scores[item_id] = scores.get(item_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.keys(), key=lambda x: -scores[x])


def search_word_plus_char(row, k=50):
    """Объединение word TF-IDF и char TF-IDF через RRF."""
    r_word = search_word_tfidf(row, k=200)
    r_char = search_char_tfidf(row, k=200)
    fused = rrf_fuse([r_word, r_char], k=60)
    return fused[:k]


# Оценка
recall_word_char, _ = evaluate_retrieval(
    eval_sample, search_word_plus_char,
    desc="Word + Char TF-IDF + RRF"
)
print(f"\n>>> Recall@50 (word + char TF-IDF, RRF): {recall_word_char:.4f}")

Char TF-IDF по категориям: 100%|██████████| 47/47 [10:09<00:00, 12.97s/it]


Готово символьных векторизаторов: 15


Word + Char TF-IDF + RRF:   0%|          | 1/5000 [00:17<23:46:59, 17.13s/it]


KeyboardInterrupt: 